In [ ]:
%pip install matplotlib numpy yfinance

In [ ]:
# Download

import yfinance as yf

def download(ticker, file):
    yf.Ticker(ticker).history(period="max", interval="1d").to_csv(file)

# download("VOO", "VOO.csv")
# download("^GSPC", "GSPC.csv")

In [ ]:
# Timings
import matplotlib.pyplot as plt
import pandas as pd

def add_drop_diff(df):
    df["Drop"] = (df["Open"] - df["Low"]) / df["Open"]
    df["Diff"] = (df["Close"] - df["Open"]) / df["Open"]

def plot_drop_diff(df, title):
    plt.scatter(df["Drop"], df["Diff"], s=1)
    plt.title(title)
    plt.xlabel("(open - low) / open")
    plt.ylabel("(close - open) / open")
    plt.show()

def plot_discount_ev(df, title):
    points = list(zip(df["Drop"], df["Diff"]))

    discounts = [0.01 * e / 1000 for e in range(1000)]
    expected_value = [sum(discount if discount < drop else -diff for drop, diff in points) / len(points) for discount in discounts]

    plt.scatter(discounts, expected_value, s=1)
    plt.title(title)
    plt.xlabel("discount")
    plt.ylabel("expected value")
    plt.show()

voo_df = pd.read_csv("VOO.csv")

add_drop_diff(voo_df)
plot_drop_diff(voo_df, "VOO (2010-2025). Data: Yahoo, 1 day = 1 point")
plot_discount_ev(voo_df, "VOO (2010-2025). Data: Yahoo, 1 day = 1 point")

In [ ]:
import pandas as pd
import json
import matplotlib.pyplot as plt
import statistics

with open("cpi.json", "r") as file:
    cpi = json.load(file)

df = pd.read_csv("GSPC.csv", parse_dates=["Date"],
    date_parser=lambda col: pd.to_datetime(col, utc=True).tz_localize(None))

df['YYYY-MM'] = df['Date'].dt.strftime('%Y-%m')
df['cpi'] = df['YYYY-MM'].map(cpi)
df = df[df["cpi"].notna()]

df["adj_open"] = df["Open"] / df["cpi"]

df = df.sort_values('Date')
df.set_index('Date', inplace=True)

def get_normalized_gains(start_dt, years):
    days = int(365.25 * years)
    roll = df['adj_open'].rolling(window=f'{days}D')

    result = pd.DataFrame({
        'first': roll.apply(lambda x: x.iloc[0], raw=False),
        'last' : roll.apply(lambda x: x.iloc[-1], raw=False)
    })

    filtered = result[result.index > pd.to_datetime(start_dt)]
    filtered["gain"] = filtered["last"] / filtered["first"]

    return [gain ** (1 / years) for gain in filtered["gain"]]

def plot(start_dt, years):
    gains = get_normalized_gains(start_dt, years)
    print(start_dt, years, sum(gains) / len(gains))
    
    pcts = statistics.quantiles(sorted(gains), n=100, method="inclusive")

    plt.plot(pcts)
    plt.xlabel("Percentile")
    plt.ylabel("Yearly normalized return")
    plt.title(f"Inflation-adjusted returns distribution of S&P500 Index\n{start_dt} - now, {years} year sliding window\n1 point = 1 percentile, Data: Yahoo, bls.gov")
    plt.show()

plot(start_dt="1975-01-01", years=10)
plot(start_dt="1975-01-01", years=20)